# Ingesta Hermética y Replicación del Orquestador BPO

El requerimiento fundamental para asegurar la validez empírica de este análisis es someter los modelos semánticos exactamente al mismo "Cortafuegos de Pasividad" que fulminó al Machine Learning clásico en el Cuaderno 02.

Recuperamos la partición Telco estratificada e instanciamos el orquestador BPO. El objetivo es interceptar las matrices de probabilidad con el umbral implacable del **0.85**. Queremos comprobar si inyectar inteligencia semántica pre-entrenada logra calibrar las predicciones lo suficiente como para empezar a automatizar tickets reales.

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import log_loss, f1_score, cohen_kappa_score
import warnings

# Bloqueamos el spam estético en la salida estándar
warnings.filterwarnings('ignore', category=UserWarning)

print("Cargando matriz Gold...")
# El motor fastparquet elude fugas de memoria en C++ que suele provocar pyarrow en Windows
df_train = pd.read_parquet('../data/gold/train_set_telco.parquet', engine='fastparquet')

# Aislamiento de vectores estáticos globales
X = df_train['full_text']
y = df_train['target_tripleta']
folds = df_train['fold_id']

print(f"Dataset cargado -> Volumen: {len(df_train)} tickets | Clases: {y.nunique()}")

def evaluar_modelo_bpo(estimator, X, y, fold_ids, umbral_confianza=0.60):
    resultados = []
    
    for fold in sorted(fold_ids.unique()):
        # Vectorizamos la máscara para evitar colisiones entre el index de Pandas y los arrays puros de Numpy
        idx_train = (fold_ids != fold).values
        idx_val = (fold_ids == fold).values
        
        # Indexación robusta (polimorfismo). Soporta Series de Pandas o tensores NumPy
        X_train = X.loc[idx_train] if hasattr(X, 'loc') else X[idx_train]
        X_val = X.loc[idx_val] if hasattr(X, 'loc') else X[idx_val]
        
        y_train = y.loc[idx_train]
        y_val = y.loc[idx_val]
        
        # Telemetría de Entrenamiento (Coste computacional)
        t0 = time.time()
        estimator.fit(X_train, y_train)
        fit_time = time.time() - t0
        
        # Telemetría de Inferencia (Latencia operativa)
        t1 = time.time()
        y_proba = estimator.predict_proba(X_val)
        latency_ms = ((time.time() - t1) / len(X_val)) * 1000
        
        y_max_proba = np.max(y_proba, axis=1)
        y_pred_bruto = estimator.classes_[np.argmax(y_proba, axis=1)]
        
        # Mapeo de Pérdida Académica (Corregido zero_division)
        ll = log_loss(y_val, y_proba, labels=estimator.classes_)
        f1_mac = f1_score(y_val, y_pred_bruto, average='macro', zero_division=0)
        f1_wei = f1_score(y_val, y_pred_bruto, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_val, y_pred_bruto)
        
        # Motor Lógico BPO (El filtro Auditor)
        mask_auto = y_max_proba >= umbral_confianza
        tasa_auto = np.mean(mask_auto)
        
        if np.sum(mask_auto) > 0:
            y_val_auto = y_val.values[mask_auto]
            y_pred_auto = y_pred_bruto[mask_auto]
            prec_cond = np.mean(y_val_auto == y_pred_auto)
        else:
            prec_cond = 0.0
            
        resultados.append({
            'Fold': fold,
            'Fit_Time_s': fit_time,
            'Latency_ms': latency_ms,
            'Log_Loss': ll,
            'F1_Macro': f1_mac,
            'F1_Weighted': f1_wei,
            'Kappa': kappa,
            'Tasa_Automatizacion': tasa_auto,
            'Precision_Condicionada': prec_cond
        })
        
    return pd.DataFrame(resultados)

print("Orquestador BPO (Motor de Evaluación) en memoria.")

Cargando matriz Gold...
Dataset cargado -> Volumen: 12322 tickets | Clases: 56
Orquestador BPO (Motor de Evaluación) en memoria.


# Extracción Estática de Embeddings (Frozen Transformer)

Abandonamos el espacio disperso del TF-IDF para proyectar la semántica de los tickets en un hiperplano continuo de 384 dimensiones. 

Inyectamos el modelo neuronal `BAAI/bge-small-en-v1.5` operando estrictamente como extractor de características estático (*Frozen*). Dado que los pesos de la red no se van a actualizar (*no hay Fine-Tuning*), la red actuará como un traductor ciego de inglés general. Queremos auditar si un modelo semántico generalista es capaz de descifrar la taxonomía específica de un BPO de telecomunicaciones sin haber sido reentrenado para ello.

In [2]:
from sentence_transformers import SentenceTransformer
import time

print("Instanciando el modelo neuronal BAAI/bge-small-en-v1.5 (Aislamiento Local y Control CPU)...")
modelo_encoder = SentenceTransformer(
    'BAAI/bge-small-en-v1.5', 
    cache_folder='../models/bge-small_local/',
    device='cpu'
)

modelo_encoder.max_seq_length = 512

print(f"Iniciando extracción vectorial para {len(X)} registros (Carga CPU Intensiva)...")
t0_embed = time.time()

# SANITIZACIÓN DE TENSORES
# Blindamos el tensor contra corrupciones de lectura del .parquet (Nulos fantasma)
X_limpio = X.fillna("").astype(str).tolist()

# Proyección estática y conversión a hiperplano denso sobre la lista purificada
X_embeddings = modelo_encoder.encode(X_limpio, show_progress_bar=True)

tiempo_total = time.time() - t0_embed
print(f"Proyección semántica completada en {tiempo_total/60:.2f} minutos.")
print(f"Morfología matemática del nuevo tensor: {X_embeddings.shape}")

d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instanciando el modelo neuronal BAAI/bge-small-en-v1.5 (Aislamiento Local y Control CPU)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4191.44it/s]


Iniciando extracción vectorial para 12322 registros (Carga CPU Intensiva)...


Batches: 100%|██████████| 386/386 [01:19<00:00,  4.85it/s]

Proyección semántica completada en 1.33 minutos.
Morfología matemática del nuevo tensor: (12322, 384)


# Modelo C: Regresión Logística sobre Espacio Semántico Denso

Comprimida la dispersión a 384 dimensiones semánticas, reinstanciamos la Regresión Logística. La hipótesis a validar es si esta nueva densidad neuronal permite a la regresión emitir curvas de probabilidad más afiladas que superen la barrera estática de confianza del 0.85.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

print("Instanciando Modelo C (Pipeline: Escalado Estándar + LogReg Densa)...")

# Transformación isométrica: StandardScaler expande la anisotropía de los tensores
# para que el solucionador lbfgs encuentre gradientes viables.
pipeline_semantico = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight='balanced', 
        max_iter=1000, 
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    )
)

print("Iniciando orquestación de Cross-Validation sobre la matriz neuronal...")
df_resultados_semanticos = evaluar_modelo_bpo(
    estimator=pipeline_semantico, 
    X=X_embeddings, 
    y=y, 
    fold_ids=folds, 
    umbral_confianza=0.85
)

print("\n--- Telemetría Consolidada: Modelo C Corregido ---")
metricas_promedio_semantico = df_resultados_semanticos.mean().drop('Fold')
print(metricas_promedio_semantico.round(4))

Instanciando Modelo C (Pipeline: Escalado Estándar + LogReg Densa)...
Iniciando orquestación de Cross-Validation sobre la matriz neuronal...


d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.


--- Telemetría Consolidada: Modelo C Corregido ---
Fit_Time_s                3.8288
Latency_ms                0.0021
Log_Loss                  4.1690
F1_Macro                  0.0930
F1_Weighted               0.1086
Kappa                     0.0584
Tasa_Automatizacion       0.0875
Precision_Condicionada    0.3525
dtype: float64


# Síntesis Operativa y El Riesgo del SLA Suicida

Los resultados de este Modelo C (Embeddings Congelados + ML Clásico) son un desastre operativo de manual y la prueba irrefutable de que usar modelos sin *Fine-Tuning* es un riesgo inasumible en entornos empresariales:

1. **Colapso de Representación (F1 = 0.09):** El Transformer sabe inglés general, pero no entiende las fronteras de decisión topológicas de un BPO Telco. Al no adaptar sus pesos, el modelo es incapaz de discriminar semánticamente las colas.
2. **El Temerario Suicida (El SLA destruido):** A diferencia del Random Forest (que dudaba siempre y automatizaba 0%), este Modelo C logra automatizar un **8.75%** del volumen al umbral del 0.85. Sin embargo, su Precisión Condicionada se desploma al **35.25%**. Esto significa que cuando el modelo se declara hiper-confiado, **se equivoca el 65% de las veces**.
3. **Conclusión Arquitectónica:** Poner esto en producción enrutará erróneamente casi 1 de cada 10 tickets diarios, colapsando el SLA. Queda demostrado que la única vía matemática viable es el *Fine-Tuning End-to-End*: necesitamos actualizar los gradientes del Encoder (RoBERTa) para que aprenda el contexto Telco y alinee su confianza probabilística con la precisión real.